In [1]:
import sys
sys.path.append("../../")
import time
import pandas as pd

T0 = time.time()  # only grid-search logs written after this point are reported at the end

# NOTE: the original variable names say "stroke" but the data loaded is WiDS; names kept, columns adapted to WiDS.
df_stroke = pd.read_csv(r"../data/wids_pre/wids.csv")

In [2]:
from callmefair.search._search_base import CType, combine_attributes

combined_df = combine_attributes(df_stroke, 'race', 'payer', CType.intersection)
combined_df

Matplotlib is building the font cache; this may take a moment.


pip install 'aif360[Reductions]'


pip install 'aif360[Reductions]'


pip install 'aif360[Reductions]'


,patient_age,breast_cancer_diagnosis_code,cancer_area,metastatic_cancer_diagnosis_code,population,female,married,income_household_150_over,income_household_six_figure,income_individual_median,rent_median,education_bachelors,education_graduate,DiagPeriodL90D,neighborhood_affluence,neighborhood_education,neighborhood_employment,neighborhood_majority_white,state_privileged,race_payer
0,0,45,21,2,31437.75000,50.142857,36.571429,7.528571,19.100000,24563.57143,1165.000000,8.357143,3.257143,1,0,0,0,0,1,0
1,0,27,26,0,39121.87879,50.106061,50.245455,29.596970,49.357576,41287.27273,2003.125000,23.739394,12.245455,1,1,1,1,0,1,1
2,0,16,3,0,21996.68333,49.876667,55.753333,18.680000,39.555000,40399.03333,1235.907407,19.678333,10.115000,1,1,0,1,1,0,1
3,0,20,22,0,32795.32558,50.933333,52.604762,38.057143,56.907143,55336.28571,2354.738095,33.285714,22.459524,0,1,1,1,0,1,1
4,0,7,21,0,10886.26000,47.688000,57.882000,8.606000,22.226000,29073.18367,919.743590,13.978000,5.684000,0,0,0,0,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12818,0,27,26,0,19413.05882,51.735294,36.429412,7.358824,18.352941,27888.52941,772.647059,14.400000,8.370588,1,0,0,0,0,0,1
12819,0,44,19,0,30153.87952,50.272840,53.076543,37.245000,55.248750,52778.65000,2223.445946,26.903704,18.277778,1,1,1,1,0,1,1
12820,0,44,19,2,32795.32558,50.933333,52.604762,38.057143,56.907143,55336.28571,2354.738095,33.285714,22.459524,1,1,1,1,0,1,1
12821,0,7,21,0,71374.13158,52.331579,39.923684,21.318421,36.207895,39491.78947,1678.447368,24.371053,16.655263,0,0,1,0,0,0,1


In [3]:
combined_df.to_csv('wids_race_payer.csv',index=False)

In [4]:
from callmefair.util.fair_util import BMInterface
from callmefair.mitigation.fair_bm import BMManager

# Defining the name of the label column
label_name = 'DiagPeriodL90D'
# Define the name of the protected (combined) attribute
sensitive_attribute = ['race_payer']

# BMInterface now performs the stratified train/val/test split internally
bm_interface = BMInterface(combined_df, label_name, sensitive_attribute)

privileged_groups = [{'race_payer': 1}]
unprivileged_groups = [{'race_payer': 0}]

bm_manager = BMManager(bm_interface, privileged_groups, unprivileged_groups)

In [5]:
from pytorch_tabnet.tab_model import TabNetClassifier
from callmefair.mitigation.fair_grid import BMGridSearch
from callmefair.mitigation.fair_bm import BMType

tabnet = TabNetClassifier(seed=42)

bm_combinations = [
    [BMType.preReweighing],  # Only preprocessing
    [BMType.preDisparate],   # Only disparate impact remover
    [BMType.preReweighing, BMType.posCalibrated],  # Preprocessing + postprocessing
]

# Initialize grid search
grid_search = BMGridSearch(
    bmI=bm_interface,
    model=tabnet,
    bm_list=bm_combinations,
    privileged_group=privileged_groups,
    unprivileged_group=unprivileged_groups
)

# Run comprehensive evaluation
grid_search.run_single_sensitive()

epoch 0  | loss: 0.75957 | val_0_auc: 0.67836 |  0:00:00s


epoch 1  | loss: 0.60096 | val_0_auc: 0.7565  |  0:00:03s


epoch 2  | loss: 0.54451 | val_0_auc: 0.77238 |  0:00:05s


epoch 3  | loss: 0.50765 | val_0_auc: 0.79149 |  0:00:06s


epoch 4  | loss: 0.49396 | val_0_auc: 0.79391 |  0:00:07s


epoch 5  | loss: 0.49393 | val_0_auc: 0.79307 |  0:00:07s


... (58 outputs left out)

epoch 11 | loss: 0.48617 | val_0_auc: 0.78609 |  0:00:18s


epoch 12 | loss: 0.48302 | val_0_auc: 0.78856 |  0:00:21s


epoch 13 | loss: 0.48448 | val_0_auc: 0.79376 |  0:00:22s


epoch 14 | loss: 0.48101 | val_0_auc: 0.79115 |  0:00:22s

Early stopping occurred at epoch 14 with best_epoch = 4 and best_val_0_auc = 0.79391


---

In [6]:
# Single attribute for comparison ('race' alone, instead of the race AND payer intersection)
sensitive_attribute = ['race']
privileged_groups = [{'race': 1}]
unprivileged_groups = [{'race': 0}]

bm_interface = BMInterface(df_stroke, label_name, sensitive_attribute)
bm_manager = BMManager(bm_interface, privileged_groups, unprivileged_groups)

# Initialize grid search
grid_search = BMGridSearch(
    bmI=bm_interface,
    model=tabnet,
    bm_list=bm_combinations,
    privileged_group=privileged_groups,
    unprivileged_group=unprivileged_groups
)

# Run comprehensive evaluation
grid_search.run_single_sensitive()

epoch 0  | loss: 0.71614 | val_0_auc: 0.68091 |  0:00:03s


epoch 1  | loss: 0.6021  | val_0_auc: 0.74396 |  0:00:04s


epoch 2  | loss: 0.53183 | val_0_auc: 0.77129 |  0:00:04s


epoch 3  | loss: 0.50964 | val_0_auc: 0.78087 |  0:00:07s


epoch 4  | loss: 0.49478 | val_0_auc: 0.78924 |  0:00:10s


epoch 5  | loss: 0.4863  | val_0_auc: 0.77527 |  0:00:13s


... (94 outputs left out)

epoch 15 | loss: 0.47707 | val_0_auc: 0.79868 |  0:00:13s


epoch 16 | loss: 0.47798 | val_0_auc: 0.7898  |  0:00:14s


epoch 17 | loss: 0.47727 | val_0_auc: 0.78334 |  0:00:15s


epoch 18 | loss: 0.4772  | val_0_auc: 0.78431 |  0:00:16s

Early stopping occurred at epoch 18 with best_epoch = 8 and best_val_0_auc = 0.80502


In [7]:
# Show what the grid searches logged (first file: race AND payer intersection; second: race alone)
import glob, os
files = [f for f in sorted(glob.glob('results/experiment_*.csv'), key=os.path.getmtime) if os.path.getmtime(f) >= T0]
for f in files:
    d = pd.read_csv(f, index_col=0)
    d = d[d.model != 'model']
    print(f)
    display(d[['model','BM','balanced_acc','acc','spd','eq_opp_diff','avg_odd_diff','fair_score']])

results/experiment_(2026_11_05-05_56_43_PM).csv


,model,BM,balanced_acc,acc,spd,eq_opp_diff,avg_odd_diff,fair_score
0,TabNetClassifier,baseline,0.737911,0.793453,0.029893,0.007896,0.042590,0.114
1,TabNetClassifier,preReweighing,0.737911,0.793453,0.029893,0.007896,0.042590,0.114
2,TabNetClassifier,preDisparate,0.739361,0.794232,0.026475,0.009776,0.037506,0.106
3,TabNetClassifier,preReweighing posCalibrated,0.737911,0.793453,0.029893,0.007896,0.042590,0.114


results/experiment_(2026_11_05-05_59_02_PM).csv


,model,BM,balanced_acc,acc,spd,eq_opp_diff,avg_odd_diff,fair_score
0,TabNetClassifier,baseline,0.739361,0.794232,0.029394,0.009036,0.055429,0.126
1,TabNetClassifier,preReweighing,0.739361,0.794232,0.029394,0.009036,0.055429,0.126
2,TabNetClassifier,preDisparate,0.735836,0.791894,0.032908,0.014531,0.058108,0.138
3,TabNetClassifier,preReweighing posCalibrated,0.739361,0.794232,0.029394,0.009036,0.055429,0.126
